# NumPy 1 — arrays, creation, dtypes, shapes

**What's in here**
- Creating arrays: `array`, `arange`, `linspace`, `zeros/ones/full`, `eye`, `default_rng`
- Inspecting: `shape`, `ndim`, `size`, `dtype`, `itemsize`
- `astype` and dtype pitfalls: int overflow, float32 precision, NaN needs floats, integer division
- `reshape`, `ravel`, `flatten`, `.T`, `np.newaxis`
- Views vs copies (the #1 silent bug)
- Stacking: `vstack`, `hstack`, `concatenate`, `stack`, `column_stack`

In [1]:
import numpy as np
import pandas as pd

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.width", 120)
print(np.__version__, pd.__version__)

1.26.4 2.3.3


## Creating arrays
`np.array` converts a list (or nested list) into an ndarray. NumPy infers the *single* dtype that fits all elements, so mixing an int and a float gives float, and mixing in a string turns everything into strings.

In [2]:
a = np.array([1, 2, 3])
b = np.array([1, 2.5, 3])
c = np.array([[1, 2, 3], [4, 5, 6]])
s = np.array([1, "two", 3.0])           # everything becomes a string!

for name, arr in [("a", a), ("b", b), ("c", c), ("s", s)]:
    print(f"{name}: dtype={arr.dtype!s:10s} shape={arr.shape}  {arr!r}")

a: dtype=int64      shape=(3,)  array([1, 2, 3])
b: dtype=float64    shape=(3,)  array([1. , 2.5, 3. ])
c: dtype=int64      shape=(2, 3)  array([[1, 2, 3],
       [4, 5, 6]])
s: dtype=<U32       shape=(3,)  array(['1', 'two', '3.0'], dtype='<U32')


`arange(start, stop, step)` excludes the stop, like Python `range`. `linspace(start, stop, num)` *includes* the stop by default and you say how many points you want, so it's safer for floating-point grids.

**Pitfall:** `np.arange(0, 1, 0.1)` can have 10 or 11 elements depending on rounding; prefer `linspace` for float grids.

In [3]:
print(np.arange(5))                 # 0..4
print(np.arange(2, 10, 2))          # 2,4,6,8
print(np.arange(0, 1, 0.25))        # stop excluded
print(np.linspace(0, 1, 5))         # stop included, 5 points
print(len(np.arange(0, 1, 0.1)), len(np.linspace(0, 1, 11)))

[0 1 2 3 4]
[2 4 6 8]
[0.   0.25 0.5  0.75]
[0.   0.25 0.5  0.75 1.  ]
10 11


`zeros`, `ones`, `full`, `empty` take a *shape* (tuple). `eye` gives an identity matrix. `zeros_like(x)` copies shape *and* dtype of an existing array — handy for pre-allocating results.

In [4]:
print(np.zeros(3))
print(np.ones((2, 3)))
print(np.full((2, 2), np.nan))
print(np.eye(3))
x = np.array([1, 2, 3])
print(np.zeros_like(x), np.zeros_like(x).dtype)     # int, because x is int
print(np.zeros_like(x, dtype=float))

[0. 0. 0.]
[[1. 1. 1.]
 [1. 1. 1.]]
[[nan nan]
 [nan nan]]
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
[0 0 0] int64
[0. 0. 0.]


## Random numbers — use `default_rng`
The modern API is `rng = np.random.default_rng(seed)`; the legacy `np.random.seed()/np.random.rand()` still works but is discouraged. A seeded generator makes notebooks reproducible.

**Interview check:** "Why does your result change every run?" → you didn't seed, or you seeded once but the cells were run out of order.

In [5]:
rng = np.random.default_rng(42)
print(rng.normal(loc=0, scale=1, size=4))
print(rng.uniform(0, 10, size=(2, 3)))
print(rng.integers(0, 5, size=6))          # high is exclusive
print(rng.choice(["wind", "solar", "gas"], size=5))

rng2 = np.random.default_rng(42)           # same seed -> same numbers
print(np.array_equal(np.random.default_rng(42).normal(size=3), rng2.normal(size=3)))

[ 0.3047 -1.04    0.7505  0.9406]
[[0.9418 9.7562 7.6114]
 [7.8606 1.2811 4.5039]]
[2 1 0 4 3 3]
['solar' 'gas' 'solar' 'solar' 'solar']
True


## Inspecting an array
`shape` is a tuple, `ndim` is its length, `size` is the total number of elements, `itemsize` is bytes per element. `shape` of a 1-D array is `(n,)` — a 1-tuple, not `(n, 1)`. That distinction matters constantly (see notebook 2).

In [6]:
m = np.arange(12).reshape(3, 4)
print(m)
print("shape", m.shape, "ndim", m.ndim, "size", m.size, "dtype", m.dtype,
      "itemsize", m.itemsize, "bytes", m.nbytes)
v = np.arange(4)
print("1-D shape:", v.shape, "  2-D column shape:", v.reshape(-1, 1).shape)

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
shape (3, 4) ndim 2 size 12 dtype int64 itemsize 8 bytes 96
1-D shape: (4,)   2-D column shape: (4, 1)


## Loading real data into numpy
Usually you read with pandas and hand the columns to numpy via `.to_numpy()`. Here: two years of hourly GB-style power data.

In [7]:
df = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"])
cons = df["consumption_mwh"].to_numpy()
temp = df["temp_c"].to_numpy()
print(df.shape, cons.dtype, cons.shape)
print(cons[:5])

(17520, 6) float64 (17520,)
[26858.4 26177.8 26229.4 25381.3 25223. ]


## dtypes and `astype`
`astype` returns a *new* array. Converting float → int **truncates towards zero** (does not round). Use `np.round(...).astype(int)` if you want rounding.

**Pitfall:** `astype(int)` on an array containing NaN produces garbage (a huge negative number) with at most a warning.

In [8]:
f = np.array([1.9, -1.9, 2.5, 3.5])
print(f.astype(int))                 # truncation: 1, -1, 2, 3
print(np.round(f).astype(int))       # banker's rounding: 2, -2, 2, 4

with np.errstate(invalid="ignore"):
    bad = np.array([1.0, np.nan]).astype(int)
print(bad)                           # NaN -> nonsense integer, silently

[ 1 -1  2  3]
[ 2 -2  2  4]
[                   1 -9223372036854775808]


### Integer overflow
NumPy integers are fixed width. `int8` wraps at 127, `int32` wraps at ~2.1e9. Python ints never overflow, so this surprises people coming from pure Python. The default integer dtype on Linux is `int64`.

**Interview check:** "Sum of a column is negative even though every value is positive?" → overflow in a small int dtype (common after reading a CSV with `dtype=np.int32` or from a database).

In [9]:
small = np.array([100, 100], dtype=np.int8)
with np.errstate(over="ignore"):
    print(small.sum(dtype=np.int8))          # 200 wraps to -56
print(small.sum())                           # sum() upcasts to int64 by default: 200

i32 = np.array([2_000_000_000], dtype=np.int32)
with np.errstate(over="ignore"):
    print(i32 + i32)                          # wraps around
print(i32.astype(np.int64) + i32.astype(np.int64))

-56
200
[-294967296]
[4000000000]


### float32 vs float64
float32 has ~7 significant digits, float64 ~16. Accumulating many float32 values (or adding a small number to a large one) loses precision. Prices/volumes in float32 are a classic source of "why doesn't this sum match the report".

In [10]:
x64 = np.full(1_000_000, 0.1)
x32 = x64.astype(np.float32)
print("float64 sum:", x64.sum())
print("float32 sum:", x32.sum(dtype=np.float32))   # noticeably off from 100000

print(np.float32(16_777_216) + np.float32(1))       # 2**24 + 1 is not representable in float32
print(np.float64(16_777_216) + np.float64(1))

float64 sum: 99999.9999999998
float32 sum: 100000.086
16777216.0
16777217.0


### NaN needs a float dtype
There is no integer NaN. Inserting NaN into an int array either fails or forces an upcast to float. This is why an integer column in pandas silently becomes float after you introduce a missing value.

In [11]:
ints = np.array([1, 2, 3])
try:
    ints[0] = np.nan
except ValueError as e:
    print("ValueError:", e)

floats = ints.astype(float)
floats[0] = np.nan
print(floats, floats.dtype)
print("nan == nan ?", np.nan == np.nan, "  use np.isnan:", np.isnan(floats))

ValueError: cannot convert float NaN to integer
[nan  2.  3.] float64
nan == nan ? False   use np.isnan: [ True False False]


### Division
`/` always produces floats (true division), `//` is floor division and keeps ints. Division by zero gives `inf`/`nan` with a *warning*, not an exception — so it can slip through silently.

In [12]:
a = np.array([7, 8, 9])
print(a / 2, (a / 2).dtype)
print(a // 2, (a // 2).dtype)
with np.errstate(divide="ignore", invalid="ignore"):
    print(np.array([1.0, 0.0, -1.0]) / 0.0)      # inf, nan, -inf  -- no exception!

[3.5 4.  4.5] float64
[3 4 4] int64
[ inf  nan -inf]


## Reshaping
`reshape` needs the total size to match; `-1` means "work this dimension out". `ravel` returns a flat view when possible, `flatten` always copies. `.T` transposes (for 1-D arrays `.T` does nothing — another `(n,)` vs `(n,1)` trap).

In [13]:
m = np.arange(6)
print(m.reshape(2, 3))
print(m.reshape(3, -1))          # -1 inferred as 2
print(m.reshape(2, 3).T)         # transpose
print(m.reshape(2, 3).ravel())   # back to 1-D
print("1-D .T does nothing:", m.T.shape, "  reshape(-1,1).T:", m.reshape(-1, 1).T.shape)

[[0 1 2]
 [3 4 5]]
[[0 1]
 [2 3]
 [4 5]]
[[0 3]
 [1 4]
 [2 5]]
[0 1 2 3 4 5]
1-D .T does nothing: (6,)   reshape(-1,1).T: (1, 6)


Row-major (C) order is the default: the *last* axis varies fastest. `order="F"` fills column-wise. Matters when you reshape time-series that were stored column-per-day etc.

In [14]:
print(np.arange(6).reshape(2, 3, order="C"))
print(np.arange(6).reshape(2, 3, order="F"))
# e.g. 48 hours -> (2 days, 24 hours): each row is one day
hours = cons[:48].reshape(2, 24)
print(hours.shape, hours.mean(axis=1).round(0))   # mean per day

[[0 1 2]
 [3 4 5]]
[[0 2 4]
 [1 3 5]]
(2, 24) [30587. 30375.]


## Views vs copies — the silent bug
Basic slicing (`a[2:5]`, `a[::2]`, `a[:, 0]`) returns a **view**: modifying it modifies the original. Fancy indexing (`a[[0, 2]]`) and boolean masks return **copies**. Use `.copy()` when you intend to mutate a slice independently. `np.shares_memory` tells you which one you have.

**Interview check:** "You normalised a slice and the original data changed. Why?" → the slice was a view.

In [15]:
a = np.arange(10, dtype=float)
view = a[::2]
view[:] = -1                      # writes through to a
print("a after modifying view:", a)

a = np.arange(10, dtype=float)
fancy = a[[0, 2, 4]]
fancy[:] = -1                     # a untouched
print("a after modifying fancy copy:", a)

print("shares memory? slice:", np.shares_memory(a, a[::2]),
      " fancy:", np.shares_memory(a, a[[0, 2]]),
      " copy:", np.shares_memory(a, a[::2].copy()))

a after modifying view: [-1.  1. -1.  3. -1.  5. -1.  7. -1.  9.]
a after modifying fancy copy: [0. 1. 2. 3. 4. 5. 6. 7. 8. 9.]
shares memory? slice: True  fancy: False  copy: False


`reshape`, `ravel`, `.T` also return views when they can. `a.base is not None` means `a` is a view of something. And note that arithmetic (`a * 2`) always creates a new array, while in-place ops (`a *= 2`) modify the existing one — which matters if `a` is itself a view of your DataFrame's data.

In [16]:
a = np.arange(6)
r = a.reshape(2, 3)
r[0, 0] = 99
print(a, "  r.base is a:", r.base is a)

b = np.arange(3, dtype=float)
c = b * 2          # new array
b *= 2             # in place
print(c, b, np.shares_memory(b, c))

[99  1  2  3  4  5]   r.base is a: True
[0. 2. 4.] [0. 2. 4.] False


## `np.newaxis` / `None` — adding a dimension
`x[:, np.newaxis]` turns `(n,)` into `(n, 1)`; `x[np.newaxis, :]` into `(1, n)`. This is the standard way to prepare arrays for broadcasting or for sklearn, which wants 2-D `X`.

In [17]:
x = np.arange(4)
print(x.shape, x[:, np.newaxis].shape, x[np.newaxis, :].shape, x[:, None].shape)

# sklearn needs X 2-D:  (n_samples, n_features)
X = temp[:, np.newaxis]
print(X.shape, "  vs  temp.reshape(-1, 1):", temp.reshape(-1, 1).shape)

(4,) (4, 1) (1, 4) (4, 1)
(17520, 1)   vs  temp.reshape(-1, 1): (17520, 1)


## Stacking arrays
- `concatenate` joins along an *existing* axis
- `stack` creates a *new* axis
- `vstack` = stack rows (axis 0), `hstack` = stack columns for 2-D / append for 1-D
- `column_stack` turns several 1-D arrays into columns — the usual way to build a feature matrix by hand

**Pitfall:** `hstack` on 1-D arrays *appends* them (gives a longer 1-D array), it does not make columns. Use `column_stack` for that.

In [18]:
a = np.array([1, 2, 3]); b = np.array([4, 5, 6])
print("concatenate:", np.concatenate([a, b]))
print("stack axis0:\n", np.stack([a, b]))          # shape (2, 3)
print("stack axis1:\n", np.stack([a, b], axis=1))  # shape (3, 2)
print("vstack:\n", np.vstack([a, b]))
print("hstack (1-D appends!):", np.hstack([a, b]))
print("column_stack:\n", np.column_stack([a, b]))

concatenate: [1 2 3 4 5 6]
stack axis0:
 [[1 2 3]
 [4 5 6]]
stack axis1:
 [[1 4]
 [2 5]
 [3 6]]
vstack:
 [[1 2 3]
 [4 5 6]]
hstack (1-D appends!): [1 2 3 4 5 6]
column_stack:
 [[1 4]
 [2 5]
 [3 6]]


Realistic use: build a design matrix from several columns and check the shape before fitting anything. Always print shapes; most numpy bugs are shape bugs.

In [19]:
hour = df["time"].dt.hour.to_numpy()
X = np.column_stack([np.ones(len(df)), temp, hour])
print(X.shape, X[:3])
X2 = np.concatenate([X, df[["wind_ms", "solar_wm2"]].to_numpy()], axis=1)
print(X2.shape)

(17520, 3) [[ 1.    0.11  0.  ]
 [ 1.   -0.18  1.  ]
 [ 1.   -1.11  2.  ]]
(17520, 5)


## Quick reference

| Task | Code |
|---|---|
| float grid | `np.linspace(a, b, n)` |
| reproducible randoms | `rng = np.random.default_rng(42); rng.normal(size=n)` |
| pre-allocate like x | `np.zeros_like(x, dtype=float)` |
| round then cast | `np.round(x).astype(int)` |
| independent slice | `x[i:j].copy()` |
| column vector | `x[:, None]` or `x.reshape(-1, 1)` |
| features to matrix | `np.column_stack([f1, f2, f3])` |
| detect view | `np.shares_memory(a, b)` / `a.base is not None` |